In [70]:
# Keep repository-relative paths valid from notebook subfolders.
from pathlib import Path
import os

os.chdir(next(
    root for root in (Path.cwd(), *Path.cwd().parents)
    if (root / "notebooks").is_dir() and (root / "requirements.txt").is_file()
))

# Resolve legacy IDX paths stored in existing CSVs without rewriting the data.
def _relocated_idx_path(value):
    text = str(value).replace("\\", "/")
    old_repo = "AI-Builders-Hackhaton-2026-Backend/"
    if old_repo in text:
        text = text.split(old_repo, 1)[1]
    old_raw = "data/idx_financial_statements/"
    if text.startswith(old_raw):
        text = "data/idx_financial/raw/" + text[len(old_raw):]
    return Path(text)

from pathlib import Path
import json
import re
import pandas as pd
import numpy as np

from openpyxl import load_workbook
from tqdm.auto import tqdm

In [71]:
INPUT_FILE = Path(
    "data/idx_financial/discovery/idx_financial_metric_candidates_enriched.csv"
)

TEST_OUTPUT_FILE = Path(
    "data/idx_financial/period_mapping/idx_financial_period_mapped_test.csv"
)

FULL_OUTPUT_FILE = Path(
    "data/idx_financial/period_mapping/idx_financial_period_mapped.csv"
)

CHECKPOINT_FILE = Path(
    "data/idx_financial/period_mapping/idx_financial_period_mapping_checkpoint.csv"
)

print("Input exists:", INPUT_FILE.exists())
print("Input:", INPUT_FILE)

Input exists: True
Input: data\idx_financial_metric_candidates_enriched.csv


In [72]:
candidates_df = pd.read_csv(
    INPUT_FILE
)

print(
    "Rows:",
    len(candidates_df)
)

print(
    "Columns:",
    candidates_df.columns.tolist()
)

display(
    candidates_df.head()
)

Rows: 115393
Columns: ['ticker', 'year', 'quarter', 'metric', 'matched_keyword', 'source_label', 'source_sheet', 'row_number', 'label_column', 'numeric_candidate_count', 'numeric_candidates', 'source_file', 'source_path']


,ticker,year,quarter,metric,matched_keyword,source_label,source_sheet,row_number,label_column,numeric_candidate_count,numeric_candidates,source_file,source_path
0,ZYRX,2025,Q1,cash,kas dan setara kas,Kas dan setara kas,1210000,8,0,2,"[{""column_index"": 1, ""value"": 1888962892}, {""c...",ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
1,ZYRX,2025,Q1,total_assets,jumlah aset,Jumlah aset,1210000,128,0,2,"[{""column_index"": 1, ""value"": 396429832878}, {...",ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
2,ZYRX,2025,Q1,total_liabilities,jumlah liabilitas,Jumlah liabilitas,1210000,247,0,2,"[{""column_index"": 1, ""value"": 98116784695}, {""...",ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
3,ZYRX,2025,Q1,revenue,sales and revenue,Sales and revenue,1321000,6,3,2,"[{""column_index"": 1, ""value"": 43381104132}, {""...",ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...
4,ZYRX,2025,Q1,cash,kas dan setara kas,Kas dan setara kas,1610000,8,0,0,[],ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...


In [73]:
def parse_numeric_candidates(value):

    if pd.isna(value):
        return []

    if isinstance(value, list):
        return value

    try:
        parsed = json.loads(value)

        if isinstance(parsed, list):
            return parsed

        return []

    except Exception:
        return []

In [74]:
sample_numeric = (
    candidates_df[
        candidates_df["numeric_candidate_count"] > 0
    ]
    .iloc[0]["numeric_candidates"]
)

print(sample_numeric)

print(
    parse_numeric_candidates(
        sample_numeric
    )
)

[{"column_index": 1, "value": 1888962892}, {"column_index": 2, "value": 6272313805}]
[{'column_index': 1, 'value': 1888962892}, {'column_index': 2, 'value': 6272313805}]


In [75]:
def resolve_source_path(path_text):

    if pd.isna(path_text):
        return None

    path_text = str(path_text)

    normalized = path_text.replace(
        "\\",
        "/"
    )

    path = _relocated_idx_path(normalized)

    if path.exists():
        return path

    alternative = (
        Path.cwd()
        /
        normalized
    )

    if alternative.exists():
        return alternative

    return None

In [76]:
test_path = resolve_source_path(
    candidates_df.iloc[0]["source_path"]
)

print(test_path)
print(
    "Exists:",
    test_path.exists()
    if test_path
    else False
)

data\idx_financial_statements\Financial_Statement_2025_Q2\ZYRX\2025\Q1\ZYRX_2025_Q1_FS.xlsx
Exists: True


In [77]:
def normalize_text(value):

    if value is None:
        return ""

    text = str(value).strip().lower()

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text

In [78]:
CURRENT_PATTERNS = [
    r"currentyearinstant",
    r"current year instant",
    r"currentyearduration",
    r"current year duration",
    r"currentperiod",
    r"current period"
]

COMPARATIVE_PATTERNS = [
    r"priorendyearinstant",
    r"prior end year instant",
    r"prioryearinstant",
    r"prior year instant",
    r"prioryearduration",
    r"prior year duration",
    r"previousyear",
    r"previous year",
    r"comparative period"
]

In [79]:
def classify_period_token(value):

    text = normalize_text(value)

    if not text:
        return None

    for pattern in CURRENT_PATTERNS:

        if re.search(
            pattern,
            text,
            flags=re.IGNORECASE
        ):

            return "CURRENT"

    for pattern in COMPARATIVE_PATTERNS:

        if re.search(
            pattern,
            text,
            flags=re.IGNORECASE
        ):

            return "COMPARATIVE"

    return None

In [80]:
def get_column_header_context(
    worksheet,
    metric_row,
    column_index,
    column_radius=2
):

    base_excel_column = (
        int(column_index)
        + 1
    )

    # Cari dari PALING ATAS sheet
    # sampai tepat sebelum row metric
    start_row = 1

    end_row = (
        int(metric_row)
        - 1
    )

    start_column = max(
        1,
        base_excel_column
        - column_radius
    )

    end_column = min(
        worksheet.max_column,
        base_excel_column
        + column_radius
    )

    context = []

    for row_number in range(
        start_row,
        end_row + 1
    ):

        for excel_column in range(
            start_column,
            end_column + 1
        ):

            value = worksheet.cell(
                row=row_number,
                column=excel_column
            ).value

            if value is None:
                continue

            context.append({
                "row_number": row_number,

                "column_index":
                    excel_column - 1,

                "distance_from_candidate_column":
                    abs(
                        excel_column
                        - base_excel_column
                    ),

                "value":
                    value
            })

    return context

In [81]:
def detect_period_from_context(
    context,
    candidate_column
):

    matches = []

    for item in context:

        period_type = classify_period_token(
            item["value"]
        )

        if period_type is None:
            continue

        matches.append({
            "period_type":
                period_type,

            "header_value":
                item["value"],

            "header_row":
                item["row_number"],

            "header_column":
                item["column_index"],

            "column_distance":
                abs(
                    int(
                        item["column_index"]
                    )
                    -
                    int(
                        candidate_column
                    )
                )
        })

    if not matches:

        return {
            "period_type": "UNKNOWN",
            "header_value": None,
            "header_row": None,
            "header_column": None,
            "evidence_count": 0
        }

    matches = sorted(
        matches,
        key=lambda x: (
            x["column_distance"],
            -x["header_row"]
        )
    )

    best_match = matches[0]

    return {
        "period_type":
            best_match["period_type"],

        "header_value":
            best_match["header_value"],

        "header_row":
            best_match["header_row"],

        "header_column":
            best_match["header_column"],

        "evidence_count":
            len(matches)
    }

In [82]:
def map_period_for_candidate_row(
    row
):

    parsed_candidates = (
        parse_numeric_candidates(
            row["numeric_candidates"]
        )
    )

    base_result = {
        "ticker": row["ticker"],
        "year": row["year"],
        "quarter": row["quarter"],
        "metric": row["metric"],
        "source_label": row["source_label"],
        "source_sheet": row["source_sheet"],
        "row_number": row["row_number"],
        "source_file": row["source_file"],
        "source_path": row["source_path"]
    }

    if len(parsed_candidates) == 0:

        result = base_result.copy()

        result.update({
            "candidate_column": None,
            "candidate_value": None,
            "period_type": "VALUE_MISSING",
            "period_header": None,
            "period_header_row": None,
            "mapping_status": "LABEL_FOUND_VALUE_MISSING"
        })

        return [result]

    file_path = resolve_source_path(
        row["source_path"]
    )

    if file_path is None:

        result = base_result.copy()

        result.update({
            "candidate_column": None,
            "candidate_value": None,
            "period_type": "UNKNOWN",
            "period_header": None,
            "period_header_row": None,
            "mapping_status": "FILE_NOT_FOUND"
        })

        return [result]

    try:

        workbook = load_workbook(
            _relocated_idx_path(file_path),
            read_only=True,
            data_only=True
        )

        sheet_name = str(
            row["source_sheet"]
        )

        if sheet_name not in workbook.sheetnames:

            workbook.close()

            result = base_result.copy()

            result.update({
                "candidate_column": None,
                "candidate_value": None,
                "period_type": "UNKNOWN",
                "period_header": None,
                "period_header_row": None,
                "mapping_status": "SHEET_NOT_FOUND"
            })

            return [result]

        worksheet = workbook[
            sheet_name
        ]

        results = []

        for candidate in parsed_candidates:

            column_index = candidate.get(
                "column_index"
            )

            candidate_value = candidate.get(
                "value"
            )

            context = (
                get_column_header_context(
                    worksheet=worksheet,
                    metric_row=int(
                        row["row_number"]
                    ),
                    column_index=int(
                        column_index
                    ),
                    column_radius=2
                )
            )

            period_info = (
                detect_period_from_context(
                    context,
                    candidate_column=column_index
                )
            )

            result = base_result.copy()

            result.update({
                "candidate_column":
                    column_index,

                "candidate_value":
                    candidate_value,

                "period_type":
                    period_info[
                        "period_type"
                    ],

                "period_header":
                    period_info[
                        "header_value"
                    ],

                "period_header_row":
                    period_info[
                        "header_row"
                    ],

                "mapping_status":
                    (
                        "EXPLICIT_HEADER"
                        if period_info[
                            "period_type"
                        ]
                        in [
                            "CURRENT",
                            "COMPARATIVE"
                        ]
                        else
                        period_info[
                            "period_type"
                        ]
                    )
            })

            results.append(
                result
            )

        workbook.close()

        return results

    except Exception as exc:

        result = base_result.copy()

        result.update({
            "candidate_column": None,
            "candidate_value": None,
            "period_type": "UNKNOWN",
            "period_header": None,
            "period_header_row": None,
            "mapping_status":
                f"READ_ERROR: {type(exc).__name__}"
        })

        return [result]

In [83]:
TEST_TICKERS = [
    "ZYRX",
    "AMRT",
    "KLBF",
    "TLKM",
    "BBRI"
]

In [84]:
test_candidates_df = (
    candidates_df[
        candidates_df[
            "ticker"
        ].isin(
            TEST_TICKERS
        )
    ]
    .sort_values(
        [
            "ticker",
            "year",
            "quarter"
        ],
        ascending=[
            True,
            False,
            False
        ]
    )
)

display(
    test_candidates_df[
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "source_sheet",
            "row_number",
            "numeric_candidate_count"
        ]
    ].head(100)
)

,ticker,year,quarter,metric,source_sheet,row_number,numeric_candidate_count
22683,AMRT,2024,Q4,cash,1210000,8,2
22684,AMRT,2024,Q4,total_assets,1210000,128,2
22685,AMRT,2024,Q4,total_liabilities,1210000,247,2
22686,AMRT,2024,Q4,revenue,1311000,6,2
22687,AMRT,2024,Q4,cash,1610000,8,0
...,...,...,...,...,...,...,...
104908,AMRT,2022,Q2,operating_cash_flow,1510000,37,2
64035,AMRT,2022,Q1,cash,1210000,7,2
64036,AMRT,2022,Q1,total_assets,1210000,123,2
64037,AMRT,2022,Q1,total_liabilities,1210000,232,2


In [85]:
latest_reports = (
    test_candidates_df[
        [
            "ticker",
            "year",
            "quarter",
            "source_file"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "ticker",
            "year",
            "quarter"
        ],
        ascending=[
            True,
            False,
            False
        ]
    )
    .groupby(
        "ticker",
        as_index=False
    )
    .first()
)

display(
    latest_reports
)

,ticker,year,quarter,source_file
0,AMRT,2024,Q4,AMRT_2024_Q4_FS.xlsx
1,BBRI,2025,Q1,BBRI_2025_Q1_FS.xlsx
2,KLBF,2025,Q1,KLBF_2025_Q1_FS.xlsx
3,TLKM,2025,Q1,TLKM_2025_Q1_FS.xlsx
4,ZYRX,2025,Q1,ZYRX_2025_Q1_FS.xlsx


In [86]:
test_rows_df = (
    candidates_df
    .merge(
        latest_reports[
            [
                "ticker",
                "source_file"
            ]
        ],
        on=[
            "ticker",
            "source_file"
        ],
        how="inner"
    )
    .copy()
)

print(
    "Test candidate rows:",
    len(test_rows_df)
)

display(
    test_rows_df[
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "source_sheet",
            "row_number",
            "numeric_candidate_count"
        ]
    ].head(100)
)

Test candidate rows: 36


,ticker,year,quarter,metric,source_sheet,row_number,numeric_candidate_count
0,ZYRX,2025,Q1,cash,1210000,8,2
1,ZYRX,2025,Q1,total_assets,1210000,128,2
2,ZYRX,2025,Q1,total_liabilities,1210000,247,2
3,ZYRX,2025,Q1,revenue,1321000,6,2
4,ZYRX,2025,Q1,cash,1610000,8,0
5,ZYRX,2025,Q1,revenue,1616000,6,0
6,ZYRX,2025,Q1,revenue,1617000,6,0
7,TLKM,2025,Q1,cash,3210000,8,2
8,TLKM,2025,Q1,total_assets,3210000,79,2
9,TLKM,2025,Q1,total_liabilities,3210000,169,2


In [87]:
test_period_results = []

for _, row in tqdm(
    test_rows_df.iterrows(),
    total=len(test_rows_df),
    desc="Testing period mapping",
    unit="candidate"
):

    mapped_rows = (
        map_period_for_candidate_row(
            row
        )
    )

    test_period_results.extend(
        mapped_rows
    )

test_period_df = pd.DataFrame(
    test_period_results
)

print(
    "Mapped test rows:",
    len(test_period_df)
)

Testing period mapping: 100%|██████████| 36/36 [00:49<00:00,  1.38s/candidate]

Mapped test rows: 62


In [88]:
display(
    test_period_df[
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "candidate_column",
            "candidate_value",
            "period_type",
            "period_header",
            "mapping_status",
            "source_sheet",
            "row_number"
        ]
    ].head(200)
)

,ticker,year,quarter,metric,candidate_column,candidate_value,period_type,period_header,mapping_status,source_sheet,row_number
0,ZYRX,2025,Q1,cash,1.0,1.888963e+09,CURRENT,CurrentYearInstant,EXPLICIT_HEADER,1210000,8
1,ZYRX,2025,Q1,cash,2.0,6.272314e+09,COMPARATIVE,PriorEndYearInstant,EXPLICIT_HEADER,1210000,8
2,ZYRX,2025,Q1,total_assets,1.0,3.964298e+11,CURRENT,CurrentYearInstant,EXPLICIT_HEADER,1210000,128
3,ZYRX,2025,Q1,total_assets,2.0,3.924446e+11,COMPARATIVE,PriorEndYearInstant,EXPLICIT_HEADER,1210000,128
4,ZYRX,2025,Q1,total_liabilities,1.0,9.811678e+10,CURRENT,CurrentYearInstant,EXPLICIT_HEADER,1210000,247
...,...,...,...,...,...,...,...,...,...,...,...
57,BBRI,2025,Q1,operating_cash_flow,2.0,4.908485e+07,COMPARATIVE,PriorYearDuration,EXPLICIT_HEADER,4510000,74
58,AMRT,2024,Q4,gross_profit,1.0,2.536548e+07,CURRENT,CurrentYearDuration,EXPLICIT_HEADER,1311000,8
59,AMRT,2024,Q4,gross_profit,2.0,2.306612e+07,COMPARATIVE,PriorYearDuration,EXPLICIT_HEADER,1311000,8
60,AMRT,2024,Q4,operating_cash_flow,1.0,8.063130e+06,CURRENT,CurrentYearInstant,EXPLICIT_HEADER,1510000,47


In [96]:
test_mapping_summary = (
    test_period_df
    .groupby(
        [
            "metric",
            "period_type"
        ]
    )
    .size()
    .reset_index(
        name="rows"
    )
    .sort_values(
        [
            "metric",
            "rows"
        ],
        ascending=[
            True,
            False
        ]
    )
)

display(
    test_mapping_summary
)

,metric,period_type,rows
2,cash,VALUE_MISSING,5
0,cash,COMPARATIVE,4
1,cash,CURRENT,4
3,gross_profit,COMPARATIVE,3
4,gross_profit,CURRENT,3
5,operating_cash_flow,COMPARATIVE,5
6,operating_cash_flow,CURRENT,5
9,revenue,VALUE_MISSING,5
7,revenue,COMPARATIVE,4
8,revenue,CURRENT,4


In [97]:
header_summary = (
    test_period_df[
        test_period_df[
            "period_header"
        ].notna()
    ]
    .groupby(
        [
            "period_header",
            "period_type"
        ]
    )
    .size()
    .reset_index(
        name="occurrences"
    )
    .sort_values(
        "occurrences",
        ascending=False
    )
)

display(
    header_summary.head(100)
)

,period_header,period_type,occurrences
1,CurrentYearInstant,CURRENT,16
2,PriorEndYearInstant,COMPARATIVE,14
0,CurrentYearDuration,CURRENT,10
3,PriorYearDuration,COMPARATIVE,8
4,PriorYearInstant,COMPARATIVE,4


In [98]:
unknown_test_df = (
    test_period_df[
        test_period_df[
            "period_type"
        ].isin(
            [
                "UNKNOWN",
                "AMBIGUOUS"
            ]
        )
    ]
    .copy()
)

print(
    "Unknown / ambiguous:",
    len(unknown_test_df)
)

display(
    unknown_test_df[
        [
            "ticker",
            "metric",
            "candidate_column",
            "candidate_value",
            "source_sheet",
            "row_number",
            "source_label"
        ]
    ].head(100)
)

Unknown / ambiguous: 0


,ticker,metric,candidate_column,candidate_value,source_sheet,row_number,source_label


In [99]:
def inspect_unknown_context(
    mapped_df,
    original_df,
    ticker,
    metric,
    max_rows_above=40,
    column_radius=5
):

    target = mapped_df[
        (mapped_df["ticker"] == ticker)
        &
        (mapped_df["metric"] == metric)
        &
        (mapped_df["period_type"] == "UNKNOWN")
    ]

    if target.empty:
        print("No UNKNOWN rows found.")
        return

    sample = target.iloc[0]

    source_row = original_df[
        (original_df["ticker"] == sample["ticker"])
        &
        (original_df["source_file"] == sample["source_file"])
        &
        (original_df["metric"] == sample["metric"])
        &
        (original_df["source_sheet"].astype(str) == str(sample["source_sheet"]))
        &
        (original_df["row_number"] == sample["row_number"])
    ].iloc[0]

    file_path = resolve_source_path(
        source_row["source_path"]
    )

    workbook = load_workbook(
        _relocated_idx_path(file_path),
        read_only=True,
        data_only=True
    )

    worksheet = workbook[
        str(source_row["source_sheet"])
    ]

    parsed_candidates = parse_numeric_candidates(
        source_row["numeric_candidates"]
    )

    print(
        "Ticker:",
        ticker
    )

    print(
        "Metric:",
        metric
    )

    print(
        "Sheet:",
        source_row["source_sheet"]
    )

    print(
        "Metric row:",
        source_row["row_number"]
    )

    print(
        "Source label:",
        source_row["source_label"]
    )

    print("\nNumeric candidates:")
    print(parsed_candidates)

    for candidate in parsed_candidates:

        column_index = int(
            candidate["column_index"]
        )

        print(
            "\n===================================="
        )

        print(
            "Candidate column:",
            column_index
        )

        print(
            "Candidate value:",
            candidate["value"]
        )

        context = get_column_header_context(
            worksheet=worksheet,
            metric_row=int(
                source_row["row_number"]
            ),
            column_index=column_index,
            max_rows_above=max_rows_above,
            column_radius=column_radius
        )

        for item in context:

            print(
                f'row={item["row_number"]} | '
                f'col={item["column_index"]} | '
                f'value={item["value"]}'
            )

    workbook.close()

In [100]:
inspect_unknown_context(
    mapped_df=test_period_df,
    original_df=candidates_df,
    ticker="ZYRX",
    metric="total_assets"
)

No UNKNOWN rows found.


In [101]:
unknown_ocf_df = (
    test_period_df[
        (test_period_df["metric"] == "operating_cash_flow")
        &
        (test_period_df["period_type"] == "UNKNOWN")
    ]
    .copy()
)

if unknown_ocf_df.empty:

    print(
        "No UNKNOWN operating_cash_flow rows found."
    )

else:

    unknown_ocf_ticker = (
        unknown_ocf_df[
            "ticker"
        ]
        .iloc[0]
    )

    print(
        "Unknown OCF ticker:",
        unknown_ocf_ticker
    )

    inspect_unknown_context(
        mapped_df=test_period_df,
        original_df=candidates_df,
        ticker=unknown_ocf_ticker,
        metric="operating_cash_flow"
    )

No UNKNOWN operating_cash_flow rows found.


In [102]:
test_period_df.to_csv(
    TEST_OUTPUT_FILE,
    index=False
)

print(
    "Saved test:",
    TEST_OUTPUT_FILE
)

Saved test: data\idx_financial_period_mapped_test.csv


In [130]:
from datetime import datetime

MONTH_MAP = {
    "january": 1,
    "february": 2,
    "march": 3,
    "april": 4,
    "may": 5,
    "june": 6,
    "july": 7,
    "august": 8,
    "september": 9,
    "october": 10,
    "november": 11,
    "december": 12,

    "januari": 1,
    "februari": 2,
    "maret": 3,
    "april": 4,
    "mei": 5,
    "juni": 6,
    "juli": 7,
    "agustus": 8,
    "september": 9,
    "oktober": 10,
    "november": 11,
    "desember": 12,
}


def parse_header_date(value):

    if value is None:
        return None

    if isinstance(value, datetime):
        return value.date()

    text = str(value).strip().lower()

    match = re.fullmatch(
        r"(\d{1,2})\s+([a-zA-Z]+)\s+(\d{4})",
        text
    )

    if not match:
        return None

    day = int(match.group(1))
    month_text = match.group(2)
    year = int(match.group(3))

    month = MONTH_MAP.get(
        month_text
    )

    if month is None:
        return None

    try:
        return datetime(
            year,
            month,
            day
        ).date()

    except ValueError:
        return None

In [131]:
def expected_report_date(
    year,
    quarter
):

    year = int(year)

    quarter_map = {
        "Q1": (3, 31),
        "Q2": (6, 30),
        "Q3": (9, 30),
        "Q4": (12, 31),
    }

    month, day = quarter_map[
        str(quarter).upper()
    ]

    return datetime(
        year,
        month,
        day
    ).date()

In [132]:
def build_sheet_period_map(
    worksheet,
    target_columns,
    report_year,
    report_quarter,
    max_scan_rows=200,
    column_radius=2
):

    period_map = {}

    target_columns = {
        int(col)
        for col in target_columns
        if pd.notna(col)
    }

    if not target_columns:
        return period_map

    relevant_columns = set()

    for target_column in target_columns:

        for offset in range(
            -column_radius,
            column_radius + 1
        ):

            col = (
                target_column
                + offset
            )

            if col >= 0:
                relevant_columns.add(
                    col
                )

    max_row_to_scan = min(
        worksheet.max_row,
        max_scan_rows
    )

    explicit_tokens = []
    date_tokens = []

    for row_number, row_values in enumerate(
        worksheet.iter_rows(
            min_row=1,
            max_row=max_row_to_scan,
            values_only=True
        ),
        start=1
    ):

        for column_index in relevant_columns:

            if column_index >= len(
                row_values
            ):
                continue

            value = row_values[
                column_index
            ]

            if value is None:
                continue

            # 1. Explicit period token
            period_type = (
                classify_period_token(
                    value
                )
            )

            if period_type is not None:

                explicit_tokens.append({
                    "column_index":
                        column_index,

                    "row_number":
                        row_number,

                    "period_type":
                        period_type,

                    "header_value":
                        value
                })

            # 2. Date fallback
            parsed_date = (
                parse_header_date(
                    value
                )
            )

            if parsed_date is not None:

                date_tokens.append({
                    "column_index":
                        column_index,

                    "row_number":
                        row_number,

                    "date":
                        parsed_date,

                    "header_value":
                        value
                })

    expected_date = (
        expected_report_date(
            report_year,
            report_quarter
        )
    )

    for target_column in target_columns:

        # =====================
        # PRIORITY 1:
        # EXPLICIT TOKEN
        # =====================

        possible_explicit = []

        for token in explicit_tokens:

            distance = abs(
                token["column_index"]
                -
                target_column
            )

            if distance <= column_radius:

                possible_explicit.append({
                    "distance":
                        distance,

                    **token
                })

        if possible_explicit:

            possible_explicit.sort(
                key=lambda x: (
                    x["distance"],
                    -x["row_number"]
                )
            )

            best = (
                possible_explicit[0]
            )

            period_map[
                target_column
            ] = {
                "period_type":
                    best[
                        "period_type"
                    ],

                "header_value":
                    best[
                        "header_value"
                    ],

                "header_row":
                    best[
                        "row_number"
                    ],

                "mapping_method":
                    "EXPLICIT_TOKEN"
            }

            continue

        # =====================
        # PRIORITY 2:
        # DATE HEADER
        # =====================

        possible_dates = []

        for token in date_tokens:

            distance = abs(
                token["column_index"]
                -
                target_column
            )

            if distance <= column_radius:

                possible_dates.append({
                    "distance":
                        distance,

                    **token
                })

        if not possible_dates:
            continue

        possible_dates.sort(
            key=lambda x: (
                x["distance"],
                -x["row_number"]
            )
        )

        best = (
            possible_dates[0]
        )

        if best["date"] == expected_date:

            period_type = "CURRENT"

        elif best["date"] < expected_date:

            period_type = "COMPARATIVE"

        else:

            period_type = "UNKNOWN"

        period_map[
            target_column
        ] = {
            "period_type":
                period_type,

            "header_value":
                best[
                    "header_value"
                ],

            "header_row":
                best[
                    "row_number"
                ],

            "mapping_method":
                "DATE_HEADER"
        }

    return period_map

In [111]:
test_period_map_results = []

test_files_df = (
    candidates_df[
        candidates_df[
            "ticker"
        ].isin(
            [
                "ZYRX",
                "AMRT",
                "KLBF",
                "TLKM",
                "BBRI"
            ]
        )
    ]
)

print(
    "Test candidate rows:",
    len(test_files_df)
)

Test candidate rows: 646


In [133]:
test_full_period_results = []

grouped_test_files = (
    test_files_df
    .groupby(
        [
            "source_file",
            "source_path"
        ],
        sort=False
    )
)

total_test_files = grouped_test_files.ngroups

for (
    source_file,
    source_path
), file_candidates_df in tqdm(
    grouped_test_files,
    total=total_test_files,
    desc="Testing optimized mapper",
    unit="file"
):

    file_path = resolve_source_path(
        source_path
    )

    if file_path is None:
        continue

    workbook = load_workbook(
        _relocated_idx_path(file_path),
        read_only=True,
        data_only=True
    )

    sheet_period_maps = {}

    required_sheets = (
        file_candidates_df[
            "source_sheet"
        ]
        .astype(str)
        .unique()
    )

    for sheet_name in required_sheets:

        if sheet_name not in workbook.sheetnames:
            continue

        worksheet = workbook[
            sheet_name
        ]

        sheet_rows = (
            file_candidates_df[
                file_candidates_df[
                    "source_sheet"
                ].astype(str)
                == sheet_name
            ]
        )

        target_columns = set()

        for numeric_json in (
            sheet_rows[
                "numeric_candidates"
            ]
        ):

            parsed = (
                parse_numeric_candidates(
                    numeric_json
                )
            )

            for candidate in parsed:

                column_index = (
                    candidate.get(
                        "column_index"
                    )
                )

                if column_index is not None:
                    target_columns.add(
                        int(column_index)
                    )

        sheet_period_maps[
            sheet_name
        ] = build_sheet_period_map(
            worksheet=worksheet,
            target_columns=target_columns,
            report_year=file_candidates_df["year"].iloc[0],
            report_quarter=file_candidates_df["quarter"].iloc[0],
            max_scan_rows=200
        )

    for _, row in file_candidates_df.iterrows():

        parsed_candidates = (
            parse_numeric_candidates(
                row["numeric_candidates"]
            )
        )

        base_result = {
            "ticker": row["ticker"],
            "year": row["year"],
            "quarter": row["quarter"],
            "metric": row["metric"],
            "source_label": row["source_label"],
            "source_sheet": row["source_sheet"],
            "row_number": row["row_number"],
            "source_file": row["source_file"],
            "source_path": row["source_path"]
        }

        if len(parsed_candidates) == 0:

            result = base_result.copy()

            result.update({
                "candidate_column": None,
                "candidate_value": None,
                "period_type": "VALUE_MISSING",
                "period_header": None,
                "period_header_row": None,
                "mapping_status":
                    "LABEL_FOUND_VALUE_MISSING"
            })

            test_full_period_results.append(
                result
            )

            continue

        sheet_name = str(
            row["source_sheet"]
        )

        period_map = (
            sheet_period_maps.get(
                sheet_name,
                {}
            )
        )

        for candidate in parsed_candidates:

            column_index = int(
                candidate["column_index"]
            )

            candidate_value = (
                candidate["value"]
            )

            period_info = (
                period_map.get(
                    column_index
                )
            )

            result = base_result.copy()

            if period_info is None:

                result.update({
                    "candidate_column":
                        column_index,

                    "candidate_value":
                        candidate_value,

                    "period_type":
                        "UNKNOWN",

                    "period_header":
                        None,

                    "period_header_row":
                        None,

                    "mapping_status":
                        "HEADER_NOT_FOUND"
                })

            else:

                result.update({
                    "candidate_column":
                        column_index,

                    "candidate_value":
                        candidate_value,

                    "period_type":
                        period_info[
                            "period_type"
                        ],

                    "period_header":
                        period_info[
                            "header_value"
                        ],

                    "period_header_row":
                        period_info[
                            "header_row"
                        ],

                    "mapping_status":
                        "EXPLICIT_HEADER"
                })

            test_full_period_results.append(
                result
            )

    workbook.close()

Testing optimized mapper: 100%|██████████| 97/97 [00:41<00:00,  2.33file/s]


In [134]:
test_optimized_df = pd.DataFrame(
    test_full_period_results
)

print(
    "Mapped test rows:",
    len(test_optimized_df)
)

Mapped test rows: 1152


In [135]:
test_optimized_summary = (
    test_optimized_df
    .groupby(
        [
            "metric",
            "period_type"
        ]
    )
    .size()
    .reset_index(
        name="rows"
    )
    .sort_values(
        [
            "metric",
            "rows"
        ],
        ascending=[
            True,
            False
        ]
    )
)

display(
    test_optimized_summary
)

,metric,period_type,rows
0,cash,COMPARATIVE,78
1,cash,CURRENT,78
2,cash,VALUE_MISSING,47
3,gross_profit,COMPARATIVE,59
4,gross_profit,CURRENT,59
5,operating_cash_flow,COMPARATIVE,97
6,operating_cash_flow,CURRENT,97
9,revenue,VALUE_MISSING,93
7,revenue,COMPARATIVE,78
8,revenue,CURRENT,78


In [ ]:
# unknown_breakdown = (
#     test_optimized_df[
#         test_optimized_df["period_type"] == "UNKNOWN"
#     ]
#     .groupby(
#         [
#             "ticker",
#             "year",
#             "quarter",
#             "source_sheet"
#         ]
#     )
#     .size()
#     .reset_index(
#         name="unknown_rows"
#     )
#     .sort_values(
#         "unknown_rows",
#         ascending=False
#     )
# )

# display(
#     unknown_breakdown.head(100)
# )

,ticker,year,quarter,source_sheet,unknown_rows
0,AMRT,2020,Q1,1210000,6
3,AMRT,2020,Q2,1210000,6
9,AMRT,2020,Q4,1210000,6
6,AMRT,2020,Q3,1210000,6
30,AMRT,2022,Q3,1210000,6
...,...,...,...,...,...
53,BBRI,2021,Q1,4220000,4
65,BBRI,2022,Q3,4220000,4
55,BBRI,2021,Q2,4220000,4
61,BBRI,2022,Q1,4220000,4


In [ ]:
# unknown_status = (
#     test_optimized_df[
#         test_optimized_df["period_type"] == "UNKNOWN"
#     ]
#     ["mapping_status"]
#     .value_counts()
# )

# display(
#     unknown_status
# )

mapping_status
HEADER_NOT_FOUND    728
Name: count, dtype: int64

In [ ]:
# unknown_sample = (
#     test_optimized_df[
#         test_optimized_df["period_type"] == "UNKNOWN"
#     ]
#     .iloc[0]
# )

# display(
#     unknown_sample.to_frame(
#         name="value"
#     )
# )

,value
ticker,ZYRX
year,2023
quarter,Q1
metric,cash
source_label,Kas dan setara kas
source_sheet,1210000
row_number,7
source_file,ZYRX_2023_Q1_FS.xlsx
source_path,data\idx_financial_statements\Financial_Statem...
candidate_column,1.0


In [ ]:
# def inspect_period_headers(
#     row,
#     original_df,
#     max_rows=250
# ):

#     source_match = original_df[
#         (original_df["ticker"] == row["ticker"])
#         &
#         (original_df["source_file"] == row["source_file"])
#         &
#         (
#             original_df["source_sheet"]
#             .astype(str)
#             == str(row["source_sheet"])
#         )
#         &
#         (
#             original_df["row_number"]
#             == row["row_number"]
#         )
#         &
#         (
#             original_df["metric"]
#             == row["metric"]
#         )
#     ]

#     if source_match.empty:
#         print("Original candidate not found.")
#         return

#     source_row = source_match.iloc[0]

#     file_path = resolve_source_path(
#         source_row["source_path"]
#     )

#     workbook = load_workbook(
#         file_path,
#         read_only=True,
#         data_only=True
#     )

#     sheet_name = str(
#         source_row["source_sheet"]
#     )

#     worksheet = workbook[
#         sheet_name
#     ]

#     print("Ticker:", source_row["ticker"])
#     print("Year:", source_row["year"])
#     print("Quarter:", source_row["quarter"])
#     print("Metric:", source_row["metric"])
#     print("Sheet:", sheet_name)
#     print("Source file:", source_row["source_file"])

#     print("\nPossible period/header texts:\n")

#     keywords = (
#         r"current"
#         r"|prior"
#         r"|previous"
#         r"|year"
#         r"|period"
#         r"|duration"
#         r"|instant"
#         r"|tahun"
#         r"|periode"
#     )

#     max_row_to_scan = min(
#         worksheet.max_row,
#         max_rows
#     )

#     for row_number, values in enumerate(
#         worksheet.iter_rows(
#             min_row=1,
#             max_row=max_row_to_scan,
#             values_only=True
#         ),
#         start=1
#     ):

#         for column_index, value in enumerate(
#             values
#         ):

#             if not isinstance(value, str):
#                 continue

#             if re.search(
#                 keywords,
#                 value,
#                 flags=re.IGNORECASE
#             ):

#                 print(
#                     f"row={row_number} | "
#                     f"col={column_index} | "
#                     f"value={value}"
#                 )

#     workbook.close()

In [ ]:
# inspect_period_headers(
#     row=unknown_sample,
#     original_df=candidates_df
# )

Ticker: ZYRX
Year: 2023
Quarter: Q1
Metric: cash
Sheet: 1210000
Source file: ZYRX_2023_Q1_FS.xlsx

Possible period/header texts:

row=1 | col=0 | value=[1210000] Statement of financial position presented using current and non-current - General Industry
row=6 | col=3 | value=Current assets
row=10 | col=3 | value=Current restricted funds
row=11 | col=3 | value=Current financial assets
row=12 | col=3 | value=Current financial assets at fair value through profit or loss
row=13 | col=3 | value=Current financial assets fair value through other comprehensive income
row=14 | col=3 | value=Current financial assets amortized cost investments
row=15 | col=3 | value=Other current financial assets
row=16 | col=3 | value=Current derivative financial assets
row=20 | col=3 | value=Current finance lease receivables
row=28 | col=3 | value=Current customer receivables
row=29 | col=3 | value=Current customer receivables third parties
row=30 | col=3 | value=Current customer receivables related parties
row=

In [ ]:
# def inspect_top_rows(
#     row,
#     original_df,
#     max_rows=12
# ):

#     source_match = original_df[
#         (original_df["ticker"] == row["ticker"])
#         &
#         (original_df["source_file"] == row["source_file"])
#         &
#         (
#             original_df["source_sheet"]
#             .astype(str)
#             == str(row["source_sheet"])
#         )
#         &
#         (
#             original_df["row_number"]
#             == row["row_number"]
#         )
#         &
#         (
#             original_df["metric"]
#             == row["metric"]
#         )
#     ]

#     if source_match.empty:
#         print("Original candidate not found.")
#         return

#     source_row = source_match.iloc[0]

#     file_path = resolve_source_path(
#         source_row["source_path"]
#     )

#     workbook = load_workbook(
#         file_path,
#         read_only=True,
#         data_only=True
#     )

#     sheet_name = str(
#         source_row["source_sheet"]
#     )

#     worksheet = workbook[
#         sheet_name
#     ]

#     print("Ticker:", source_row["ticker"])
#     print("Year:", source_row["year"])
#     print("Quarter:", source_row["quarter"])
#     print("Metric:", source_row["metric"])
#     print("Sheet:", sheet_name)

#     print("\nTOP ROWS RAW:\n")

#     for row_number, values in enumerate(
#         worksheet.iter_rows(
#             min_row=1,
#             max_row=min(
#                 worksheet.max_row,
#                 max_rows
#             ),
#             values_only=True
#         ),
#         start=1
#     ):

#         print(
#             f"ROW {row_number}:",
#             list(values)
#         )

#     workbook.close()

In [ ]:
# inspect_top_rows(
#     row=unknown_sample,
#     original_df=candidates_df,
#     max_rows=12
# )

Ticker: ZYRX
Year: 2023
Quarter: Q1
Metric: cash
Sheet: 1210000

TOP ROWS RAW:

ROW 1: ['[1210000] Statement of financial position presented using current and non-current - General Industry', None, None, None]
ROW 2: [None, None, None, None]
ROW 3: ['Laporan posisi keuangan', None, 'Statement of financial position', None]
ROW 4: [None, '31 March 2023', '31 December 2022', None]
ROW 5: ['Aset', None, None, 'Assets']
ROW 6: ['Aset lancar', None, None, 'Current assets']
ROW 7: ['Kas dan setara kas', 84529760967.0, 206376346678.0, 'Cash and cash equivalents']
ROW 8: ['Wesel tagih', '', '', 'Notes receivable']
ROW 9: ['Investasi jangka pendek', '', '', 'Short-term investments']
ROW 10: ['Dana yang dibatasi penggunaannya lancar', '', '', 'Current restricted funds']
ROW 11: ['Aset keuangan lancar', None, None, 'Current financial assets']
ROW 12: ['Aset keuangan lancar yang diukur pada nilai wajar melalui laba rugi', '', '', 'Current financial assets at fair value through profit or loss']


In [136]:
full_period_results = []

CHECKPOINT_EVERY_FILES = 250

grouped_files = (
    candidates_df
    .groupby(
        [
            "source_file",
            "source_path"
        ],
        sort=False
    )
)

total_files = grouped_files.ngroups

for file_index, (
    (source_file, source_path),
    file_candidates_df
) in enumerate(
    tqdm(
        grouped_files,
        total=total_files,
        desc="Mapping periods by file",
        unit="file"
    ),
    start=1
):

    file_path = resolve_source_path(
        source_path
    )

    # =========================================================
    # FILE TIDAK DITEMUKAN
    # =========================================================
    if file_path is None:

        for _, row in file_candidates_df.iterrows():

            parsed_candidates = (
                parse_numeric_candidates(
                    row["numeric_candidates"]
                )
            )

            base_result = {
                "ticker": row["ticker"],
                "year": row["year"],
                "quarter": row["quarter"],
                "metric": row["metric"],
                "source_label": row["source_label"],
                "source_sheet": row["source_sheet"],
                "row_number": row["row_number"],
                "source_file": row["source_file"],
                "source_path": row["source_path"]
            }

            # Label ada tapi angka tidak ada
            if len(parsed_candidates) == 0:

                result = base_result.copy()

                result.update({
                    "candidate_column": None,
                    "candidate_value": None,
                    "period_type": "VALUE_MISSING",
                    "period_header": None,
                    "period_header_row": None,
                    "mapping_status":
                        "LABEL_FOUND_VALUE_MISSING"
                })

                full_period_results.append(
                    result
                )

            else:

                for candidate in parsed_candidates:

                    result = base_result.copy()

                    result.update({
                        "candidate_column":
                            candidate.get(
                                "column_index"
                            ),

                        "candidate_value":
                            candidate.get(
                                "value"
                            ),

                        "period_type":
                            "UNKNOWN",

                        "period_header":
                            None,

                        "period_header_row":
                            None,

                        "mapping_status":
                            "FILE_NOT_FOUND"
                    })

                    full_period_results.append(
                        result
                    )

        continue

    # =========================================================
    # OPEN WORKBOOK
    # =========================================================
    try:

        workbook = load_workbook(
            _relocated_idx_path(file_path),
            read_only=True,
            data_only=True
        )

        # =====================================================
        # CACHE PERIOD MAP PER SHEET
        # =====================================================

        sheet_period_maps = {}

        required_sheets = (
            file_candidates_df[
                "source_sheet"
            ]
            .astype(str)
            .unique()
        )

        # Satu report/file punya year dan quarter yang sama
        report_year = (
            file_candidates_df[
                "year"
            ]
            .iloc[0]
        )

        report_quarter = (
            file_candidates_df[
                "quarter"
            ]
            .iloc[0]
        )

        for sheet_name in required_sheets:

            if sheet_name not in workbook.sheetnames:
                continue

            worksheet = workbook[
                sheet_name
            ]

            # Candidate yang berasal dari sheet ini saja
            sheet_rows = (
                file_candidates_df[
                    file_candidates_df[
                        "source_sheet"
                    ].astype(str)
                    == sheet_name
                ]
            )

            # Cari numeric columns yang memang kita butuhkan
            target_columns = set()

            for numeric_json in (
                sheet_rows[
                    "numeric_candidates"
                ]
            ):

                parsed = (
                    parse_numeric_candidates(
                        numeric_json
                    )
                )

                for candidate in parsed:

                    column_index = (
                        candidate.get(
                            "column_index"
                        )
                    )

                    if column_index is not None:

                        target_columns.add(
                            int(
                                column_index
                            )
                        )

            # HYBRID PERIOD MAPPER
            # 1. explicit token
            # 2. fallback date header
            sheet_period_maps[
                sheet_name
            ] = build_sheet_period_map(
                worksheet=worksheet,
                target_columns=target_columns,
                report_year=report_year,
                report_quarter=report_quarter,
                max_scan_rows=200,
                column_radius=2
            )

        # =====================================================
        # PROCESS SEMUA CANDIDATE DALAM FILE
        # =====================================================

        for _, row in file_candidates_df.iterrows():

            parsed_candidates = (
                parse_numeric_candidates(
                    row["numeric_candidates"]
                )
            )

            base_result = {
                "ticker": row["ticker"],
                "year": row["year"],
                "quarter": row["quarter"],
                "metric": row["metric"],
                "source_label": row["source_label"],
                "source_sheet": row["source_sheet"],
                "row_number": row["row_number"],
                "source_file": row["source_file"],
                "source_path": row["source_path"]
            }

            # =================================================
            # LABEL ADA, VALUE TIDAK ADA
            # =================================================

            if len(parsed_candidates) == 0:

                result = base_result.copy()

                result.update({
                    "candidate_column":
                        None,

                    "candidate_value":
                        None,

                    "period_type":
                        "VALUE_MISSING",

                    "period_header":
                        None,

                    "period_header_row":
                        None,

                    "mapping_status":
                        "LABEL_FOUND_VALUE_MISSING"
                })

                full_period_results.append(
                    result
                )

                continue

            sheet_name = str(
                row["source_sheet"]
            )

            # =================================================
            # SHEET TIDAK DITEMUKAN
            # =================================================

            if sheet_name not in workbook.sheetnames:

                for candidate in parsed_candidates:

                    result = base_result.copy()

                    result.update({
                        "candidate_column":
                            candidate.get(
                                "column_index"
                            ),

                        "candidate_value":
                            candidate.get(
                                "value"
                            ),

                        "period_type":
                            "UNKNOWN",

                        "period_header":
                            None,

                        "period_header_row":
                            None,

                        "mapping_status":
                            "SHEET_NOT_FOUND"
                    })

                    full_period_results.append(
                        result
                    )

                continue

            # Period map hasil cache
            period_map = (
                sheet_period_maps.get(
                    sheet_name,
                    {}
                )
            )

            # =================================================
            # MAP SETIAP NUMERIC CANDIDATE
            # =================================================

            for candidate in parsed_candidates:

                column_index = int(
                    candidate[
                        "column_index"
                    ]
                )

                candidate_value = (
                    candidate[
                        "value"
                    ]
                )

                period_info = (
                    period_map.get(
                        column_index
                    )
                )

                result = base_result.copy()

                # =============================================
                # PERIOD HEADER TIDAK DITEMUKAN
                # =============================================

                if period_info is None:

                    result.update({
                        "candidate_column":
                            column_index,

                        "candidate_value":
                            candidate_value,

                        "period_type":
                            "UNKNOWN",

                        "period_header":
                            None,

                        "period_header_row":
                            None,

                        "mapping_status":
                            "HEADER_NOT_FOUND"
                    })

                # =============================================
                # PERIOD BERHASIL DIMAPPING
                # =============================================

                else:

                    mapping_method = (
                        period_info.get(
                            "mapping_method",
                            "UNKNOWN_METHOD"
                        )
                    )

                    result.update({
                        "candidate_column":
                            column_index,

                        "candidate_value":
                            candidate_value,

                        "period_type":
                            period_info[
                                "period_type"
                            ],

                        "period_header":
                            period_info[
                                "header_value"
                            ],

                        "period_header_row":
                            period_info[
                                "header_row"
                            ],

                        "mapping_status":
                            mapping_method
                    })

                full_period_results.append(
                    result
                )

        workbook.close()

    # =========================================================
    # WORKBOOK READ ERROR
    # =========================================================

    except Exception as exc:

        for _, row in file_candidates_df.iterrows():

            parsed_candidates = (
                parse_numeric_candidates(
                    row["numeric_candidates"]
                )
            )

            base_result = {
                "ticker": row["ticker"],
                "year": row["year"],
                "quarter": row["quarter"],
                "metric": row["metric"],
                "source_label": row["source_label"],
                "source_sheet": row["source_sheet"],
                "row_number": row["row_number"],
                "source_file": row["source_file"],
                "source_path": row["source_path"]
            }

            # Kalau metric memang tidak punya angka
            if len(parsed_candidates) == 0:

                result = base_result.copy()

                result.update({
                    "candidate_column":
                        None,

                    "candidate_value":
                        None,

                    "period_type":
                        "VALUE_MISSING",

                    "period_header":
                        None,

                    "period_header_row":
                        None,

                    "mapping_status":
                        f"READ_ERROR: "
                        f"{type(exc).__name__}"
                })

                full_period_results.append(
                    result
                )

            else:

                for candidate in parsed_candidates:

                    result = base_result.copy()

                    result.update({
                        "candidate_column":
                            candidate.get(
                                "column_index"
                            ),

                        "candidate_value":
                            candidate.get(
                                "value"
                            ),

                        "period_type":
                            "UNKNOWN",

                        "period_header":
                            None,

                        "period_header_row":
                            None,

                        "mapping_status":
                            f"READ_ERROR: "
                            f"{type(exc).__name__}"
                    })

                    full_period_results.append(
                        result
                    )

    # =========================================================
    # CHECKPOINT
    # =========================================================

    if (
        file_index
        % CHECKPOINT_EVERY_FILES
        == 0
    ):

        checkpoint_df = pd.DataFrame(
            full_period_results
        )

        checkpoint_df.to_csv(
            CHECKPOINT_FILE,
            index=False
        )

        tqdm.write(
            f"Checkpoint: "
            f"{file_index:,}/"
            f"{total_files:,} files | "
            f"{len(full_period_results):,} "
            f"mapped rows"
        )

Mapping periods by file:   2%|▏         | 251/15817 [01:40<1:03:26,  4.09file/s]

Checkpoint: 250/15,817 files | 3,416 mapped rows


Mapping periods by file:   3%|▎         | 500/15817 [03:06<1:32:25,  2.76file/s]

Checkpoint: 500/15,817 files | 6,895 mapped rows


Mapping periods by file:   5%|▍         | 750/15817 [04:46<1:44:33,  2.40file/s]

...

Mapping periods by file:  98%|█████████▊| 15501/15817 [2:39:57<03:15,  1.62file/s]

Checkpoint: 15,500/15,817 files | 201,721 mapped rows


Mapping periods by file: 100%|█████████▉| 15751/15817 [2:40:49<00:35,  1.86file/s]

Checkpoint: 15,750/15,817 files | 204,329 mapped rows


Mapping periods by file: 100%|██████████| 15817/15817 [2:41:03<00:00,  1.64file/s]


In [137]:
period_mapped_df = pd.DataFrame(
    full_period_results
)

print(
    "Mapped rows:",
    len(period_mapped_df)
)

display(
    period_mapped_df.head()
)

Mapped rows: 205033


,ticker,year,quarter,metric,source_label,source_sheet,row_number,source_file,source_path,candidate_column,candidate_value,period_type,period_header,period_header_row,mapping_status
0,ZYRX,2025,Q1,cash,Kas dan setara kas,1210000,8,ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,1.0,1.888963e+09,CURRENT,CurrentYearInstant,4.0,EXPLICIT_TOKEN
1,ZYRX,2025,Q1,cash,Kas dan setara kas,1210000,8,ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,2.0,6.272314e+09,COMPARATIVE,PriorEndYearInstant,4.0,EXPLICIT_TOKEN
2,ZYRX,2025,Q1,total_assets,Jumlah aset,1210000,128,ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,1.0,3.964298e+11,CURRENT,CurrentYearInstant,4.0,EXPLICIT_TOKEN
3,ZYRX,2025,Q1,total_assets,Jumlah aset,1210000,128,ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,2.0,3.924446e+11,COMPARATIVE,PriorEndYearInstant,4.0,EXPLICIT_TOKEN
4,ZYRX,2025,Q1,total_liabilities,Jumlah liabilitas,1210000,247,ZYRX_2025_Q1_FS.xlsx,data\idx_financial_statements\Financial_Statem...,1.0,9.811678e+10,CURRENT,CurrentYearInstant,4.0,EXPLICIT_TOKEN


In [138]:
full_mapping_summary = (
    period_mapped_df
    .groupby(
        [
            "metric",
            "period_type"
        ]
    )
    .size()
    .reset_index(
        name="rows"
    )
    .sort_values(
        [
            "metric",
            "rows"
        ],
        ascending=[
            True,
            False
        ]
    )
)

display(
    full_mapping_summary
)

,metric,period_type,rows
0,cash,COMPARATIVE,14798
1,cash,CURRENT,14710
3,cash,VALUE_MISSING,8052
2,cash,UNKNOWN,98
4,gross_profit,COMPARATIVE,13445
5,gross_profit,CURRENT,13326
6,gross_profit,UNKNOWN,75
7,gross_profit,VALUE_MISSING,44
8,operating_cash_flow,COMPARATIVE,15826
9,operating_cash_flow,CURRENT,15685


In [139]:
overall_period_summary = (
    period_mapped_df[
        "period_type"
    ]
    .value_counts(
        dropna=False
    )
    .reset_index()
)

overall_period_summary.columns = [
    "period_type",
    "rows"
]

display(
    overall_period_summary
)

,period_type,rows
0,COMPARATIVE,89779
1,CURRENT,89114
2,VALUE_MISSING,25593
3,UNKNOWN,547


In [140]:
mapping_method_summary = (
    period_mapped_df[
        "mapping_status"
    ]
    .value_counts(
        dropna=False
    )
    .reset_index()
)

mapping_method_summary.columns = [
    "mapping_status",
    "rows"
]

display(
    mapping_method_summary
)

,mapping_status,rows
0,DATE_HEADER,124877
1,EXPLICIT_TOKEN,54563
2,LABEL_FOUND_VALUE_MISSING,25593


In [141]:
unknown_status_summary = (
    period_mapped_df[
        period_mapped_df["period_type"] == "UNKNOWN"
    ]
    .groupby("mapping_status")
    .size()
    .reset_index(name="rows")
    .sort_values(
        "rows",
        ascending=False
    )
)

display(unknown_status_summary)

,mapping_status,rows
0,DATE_HEADER,547


In [142]:
unknown_date_df = (
    period_mapped_df[
        period_mapped_df["period_type"] == "UNKNOWN"
    ]
    .copy()
)

print(
    "UNKNOWN rows:",
    len(unknown_date_df)
)

display(
    unknown_date_df[
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "candidate_value",
            "period_header",
            "period_header_row",
            "mapping_status",
            "source_sheet",
            "source_file"
        ]
    ].head(100)
)

UNKNOWN rows: 547


,ticker,year,quarter,metric,candidate_value,period_header,period_header_row,mapping_status,source_sheet,source_file
82091,IKBI,2023,Q1,cash,1499465.0,30 June 2023,4.0,DATE_HEADER,1210000,IKBI_2023_Q1_FS.xlsx
82093,IKBI,2023,Q1,total_assets,119780581.0,30 June 2023,4.0,DATE_HEADER,1210000,IKBI_2023_Q1_FS.xlsx
82095,IKBI,2023,Q1,total_liabilities,47125481.0,30 June 2023,4.0,DATE_HEADER,1210000,IKBI_2023_Q1_FS.xlsx
82097,IKBI,2023,Q1,revenue,60059599.0,30 June 2023,4.0,DATE_HEADER,1311000,IKBI_2023_Q1_FS.xlsx
82104,IKBI,2023,Q1,gross_profit,5848091.0,30 June 2023,4.0,DATE_HEADER,1311000,IKBI_2023_Q1_FS.xlsx
...,...,...,...,...,...,...,...,...,...,...
125542,IKBI,2022,Q3,total_liabilities,52761539.0,31 December 2022,4.0,DATE_HEADER,1210000,IKBI_2022_Q3_FS.xlsx
125544,IKBI,2022,Q3,revenue,167819952.0,31 December 2022,4.0,DATE_HEADER,1311000,IKBI_2022_Q3_FS.xlsx
125551,IKBI,2022,Q3,gross_profit,8312574.0,31 December 2022,4.0,DATE_HEADER,1311000,IKBI_2022_Q3_FS.xlsx
125553,IKBI,2022,Q3,operating_cash_flow,-4657348.0,31 December 2022,4.0,DATE_HEADER,1510000,IKBI_2022_Q3_FS.xlsx


In [143]:
unknown_header_summary = (
    unknown_date_df
    .groupby(
        [
            "year",
            "quarter",
            "period_header"
        ]
    )
    .size()
    .reset_index(
        name="rows"
    )
    .sort_values(
        "rows",
        ascending=False
    )
)

display(
    unknown_header_summary.head(100)
)

,year,quarter,period_header,rows
1,2020,Q1,30 June 2020,30
3,2020,Q1,31 December 2020,30
4,2020,Q2,30 September 2020,24
26,2022,Q1,30 June 2022,24
14,2021,Q1,30 June 2021,24
40,2023,Q1,30 June 2023,24
20,2021,Q3,31 December 2021,21
38,2022,Q4,31 March 2023,18
25,2021,Q4,31 March 2022,18
16,2021,Q2,30 September 2021,18


In [144]:
unknown_file_summary = (
    unknown_date_df
    .groupby(
        [
            "ticker",
            "year",
            "quarter",
            "source_file"
        ]
    )
    .size()
    .reset_index(
        name="rows"
    )
    .sort_values(
        "rows",
        ascending=False
    )
)

display(
    unknown_file_summary.head(100)
)

,ticker,year,quarter,source_file,rows
85,NSSS,2022,Q1,NSSS_2022_Q1_FS.xlsx,10
28,CANI,2020,Q1,CANI_2020_Q1_FS.xlsx,9
35,CANI,2022,Q1,CANI_2022_Q1_FS.xlsx,9
31,CANI,2021,Q1,CANI_2021_Q1_FS.xlsx,9
27,BUAH,2021,Q3,BUAH_2021_Q3_FS.xlsx,9
...,...,...,...,...,...
19,AMOR,2021,Q2,AMOR_2021_Q2_FS.xlsx,5
16,AMOR,2020,Q3,AMOR_2020_Q3_FS.xlsx,5
17,AMOR,2020,Q4,AMOR_2020_Q4_FS.xlsx,5
15,AMOR,2020,Q2,AMOR_2020_Q2_FS.xlsx,5


In [145]:
current_comparative_check = (
    period_mapped_df[
        period_mapped_df["period_type"].isin(
            [
                "CURRENT",
                "COMPARATIVE"
            ]
        )
    ]
    .groupby(
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "source_file"
        ]
    )["period_type"]
    .value_counts()
    .unstack(
        fill_value=0
    )
    .reset_index()
)

for col in [
    "CURRENT",
    "COMPARATIVE"
]:
    if col not in current_comparative_check.columns:
        current_comparative_check[col] = 0

imbalanced_df = (
    current_comparative_check[
        current_comparative_check[
            "CURRENT"
        ]
        !=
        current_comparative_check[
            "COMPARATIVE"
        ]
    ]
    .copy()
)

print(
    "Report/metric groups with unequal "
    "CURRENT vs COMPARATIVE:",
    len(imbalanced_df)
)

display(
    imbalanced_df.head(100)
)

Report/metric groups with unequal CURRENT vs COMPARATIVE: 785


period_type,ticker,year,quarter,metric,source_file,COMPARATIVE,CURRENT
30,AALI,2020,Q3,cash,AALI_2020_Q3_FS.xlsx,2,0
31,AALI,2020,Q3,gross_profit,AALI_2020_Q3_FS.xlsx,2,0
32,AALI,2020,Q3,operating_cash_flow,AALI_2020_Q3_FS.xlsx,2,0
33,AALI,2020,Q3,revenue,AALI_2020_Q3_FS.xlsx,2,0
34,AALI,2020,Q3,total_assets,AALI_2020_Q3_FS.xlsx,2,0
...,...,...,...,...,...,...,...
3594,AMIN,2023,Q3,cash,AMIN_2023_Q3_FS.xlsx,1,0
3595,AMIN,2023,Q3,gross_profit,AMIN_2023_Q3_FS.xlsx,1,0
3596,AMIN,2023,Q3,operating_cash_flow,AMIN_2023_Q3_FS.xlsx,1,0
3597,AMIN,2023,Q3,revenue,AMIN_2023_Q3_FS.xlsx,1,0


In [148]:
date_header_df = (
    period_mapped_df[
        period_mapped_df[
            "mapping_status"
        ].eq(
            "DATE_HEADER"
        )
    ]
    .copy()
)

date_header_df[
    "parsed_period_date"
] = (
    date_header_df[
        "period_header"
    ]
    .apply(
        parse_header_date
    )
)

print(
    "DATE_HEADER rows:",
    len(date_header_df)
)

print(
    "Unparseable header dates:",
    date_header_df[
        "parsed_period_date"
    ].isna().sum()
)

DATE_HEADER rows: 124877
Unparseable header dates: 0


In [149]:
date_group_check = (
    date_header_df
    .groupby(
        [
            "source_file",
            "source_sheet"
        ]
    )[
        "parsed_period_date"
    ]
    .nunique()
    .reset_index(
        name="unique_dates"
    )
)

display(
    date_group_check[
        "unique_dates"
    ]
    .value_counts()
    .sort_index()
)

unique_dates
1       50
2    31793
Name: count, dtype: int64

In [150]:
single_date_groups = (
    date_group_check[
        date_group_check["unique_dates"] == 1
    ]
    .copy()
)

print(
    "Single-date groups:",
    len(single_date_groups)
)

display(
    single_date_groups.head(100)
)

Single-date groups: 50


,source_file,source_sheet,unique_dates
695,AIMS_2021_Q1_FS.xlsx,1311000,1
2966,BAPI_2020_Q1_FS.xlsx,2312000,1
2969,BAPI_2020_Q2_FS.xlsx,2312000,1
2975,BAPI_2020_Q4_FS.xlsx,2311000,1
8600,DKFT_2023_Q1_FS.xlsx,1311000,1
12184,HDTX_2020_Q1_FS.xlsx,1311000,1
15086,JAYA_2023_Q1_FS.xlsx,3510000,1
15267,JKSW_2020_Q2_FS.xlsx,1311000,1
15270,JKSW_2020_Q3_FS.xlsx,1311000,1
15273,JKSW_2020_Q4_FS.xlsx,1311000,1


In [151]:
single_date_details = (
    date_header_df
    .merge(
        single_date_groups[
            [
                "source_file",
                "source_sheet"
            ]
        ],
        on=[
            "source_file",
            "source_sheet"
        ],
        how="inner"
    )
    [
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "source_file",
            "source_sheet",
            "period_header",
            "parsed_period_date",
            "candidate_column",
            "candidate_value"
        ]
    ]
)

display(
    single_date_details.head(200)
)

,ticker,year,quarter,metric,source_file,source_sheet,period_header,parsed_period_date,candidate_column,candidate_value
0,TGRA,2023,Q2,revenue,TGRA_2023_Q2_FS.xlsx,3321000,30 June 2023,2023-06-30,1.0,9.767652e+08
1,TGRA,2023,Q2,gross_profit,TGRA_2023_Q2_FS.xlsx,3321000,30 June 2023,2023-06-30,1.0,2.879135e+08
2,TGRA,2023,Q3,revenue,TGRA_2023_Q3_FS.xlsx,3321000,30 September 2023,2023-09-30,1.0,9.767652e+08
3,TGRA,2023,Q3,gross_profit,TGRA_2023_Q3_FS.xlsx,3321000,30 September 2023,2023-09-30,1.0,2.879135e+08
4,RONY,2023,Q1,revenue,RONY_2023_Q1_FS.xlsx,1311000,31 March 2022,2022-03-31,2.0,6.464545e+09
...,...,...,...,...,...,...,...,...,...,...
88,HDTX,2020,Q1,gross_profit,HDTX_2020_Q1_FS.xlsx,1311000,31 March 2020,2020-03-31,1.0,-1.019823e+07
89,BAPI,2020,Q1,revenue,BAPI_2020_Q1_FS.xlsx,2312000,31 March 2020,2020-03-31,1.0,1.622013e+09
90,BAPI,2020,Q2,revenue,BAPI_2020_Q2_FS.xlsx,2312000,30 June 2020,2020-06-30,1.0,3.344487e+09
91,BAPI,2020,Q4,revenue,BAPI_2020_Q4_FS.xlsx,2311000,31 December 2020,2020-12-31,1.0,1.327283e+10


In [153]:
period_mapped_corrected_df = (
    period_mapped_df
    .copy()
)

period_mapped_corrected_df[
    "parsed_period_date"
] = (
    period_mapped_corrected_df[
        "period_header"
    ]
    .apply(
        parse_header_date
    )
)

In [154]:
date_group_info = (
    period_mapped_corrected_df[
        period_mapped_corrected_df[
            "mapping_status"
        ].eq(
            "DATE_HEADER"
        )
        &
        period_mapped_corrected_df[
            "parsed_period_date"
        ].notna()
    ]
    .groupby(
        [
            "source_file",
            "source_sheet"
        ]
    )[
        "parsed_period_date"
    ]
    .agg(
        unique_dates="nunique",
        latest_date="max",
        earliest_date="min"
    )
    .reset_index()
)

display(
    date_group_info[
        "unique_dates"
    ]
    .value_counts()
    .sort_index()
)

unique_dates
1       50
2    31793
Name: count, dtype: int64

In [155]:
period_mapped_corrected_df = (
    period_mapped_corrected_df
    .merge(
        date_group_info,
        on=[
            "source_file",
            "source_sheet"
        ],
        how="left"
    )
)

In [156]:
two_date_mask = (
    period_mapped_corrected_df[
        "mapping_status"
    ].eq(
        "DATE_HEADER"
    )
    &
    period_mapped_corrected_df[
        "unique_dates"
    ].eq(2)
)

period_mapped_corrected_df.loc[
    two_date_mask
    &
    (
        period_mapped_corrected_df[
            "parsed_period_date"
        ]
        ==
        period_mapped_corrected_df[
            "latest_date"
        ]
    ),
    "period_type"
] = "CURRENT"

period_mapped_corrected_df.loc[
    two_date_mask
    &
    (
        period_mapped_corrected_df[
            "parsed_period_date"
        ]
        ==
        period_mapped_corrected_df[
            "earliest_date"
        ]
    ),
    "period_type"
] = "COMPARATIVE"

In [157]:
period_mapped_corrected_df.loc[
    two_date_mask,
    "mapping_status"
] = "DATE_HEADER_PAIR"

In [158]:
def get_expected_date_from_row(row):

    try:
        return expected_report_date(
            row["year"],
            row["quarter"]
        )

    except Exception:
        return None

In [159]:
period_mapped_corrected_df[
    "expected_period_date"
] = (
    period_mapped_corrected_df
    .apply(
        get_expected_date_from_row,
        axis=1
    )
)

In [160]:
single_date_mask = (
    period_mapped_corrected_df[
        "mapping_status"
    ].eq(
        "DATE_HEADER"
    )
    &
    period_mapped_corrected_df[
        "unique_dates"
    ].eq(1)
    &
    period_mapped_corrected_df[
        "parsed_period_date"
    ].notna()
    &
    period_mapped_corrected_df[
        "expected_period_date"
    ].notna()
)

In [161]:
single_current_mask = (
    single_date_mask
    &
    (
        period_mapped_corrected_df[
            "parsed_period_date"
        ]
        ==
        period_mapped_corrected_df[
            "expected_period_date"
        ]
    )
)

period_mapped_corrected_df.loc[
    single_current_mask,
    "period_type"
] = "CURRENT"

period_mapped_corrected_df.loc[
    single_current_mask,
    "mapping_status"
] = "DATE_HEADER_SINGLE_CURRENT"

In [162]:
single_comparative_mask = (
    single_date_mask
    &
    (
        period_mapped_corrected_df[
            "parsed_period_date"
        ]
        <
        period_mapped_corrected_df[
            "expected_period_date"
        ]
    )
)

period_mapped_corrected_df.loc[
    single_comparative_mask,
    "period_type"
] = "COMPARATIVE"

period_mapped_corrected_df.loc[
    single_comparative_mask,
    "mapping_status"
] = "DATE_HEADER_SINGLE_COMPARATIVE"

In [163]:
single_unknown_mask = (
    single_date_mask
    &
    (
        period_mapped_corrected_df[
            "parsed_period_date"
        ]
        >
        period_mapped_corrected_df[
            "expected_period_date"
        ]
    )
)

period_mapped_corrected_df.loc[
    single_unknown_mask,
    "period_type"
] = "UNKNOWN"

period_mapped_corrected_df.loc[
    single_unknown_mask,
    "mapping_status"
] = "DATE_HEADER_SINGLE_FUTURE"

In [164]:
corrected_period_summary = (
    period_mapped_corrected_df[
        "period_type"
    ]
    .value_counts(
        dropna=False
    )
    .reset_index()
)

corrected_period_summary.columns = [
    "period_type",
    "rows"
]

display(
    corrected_period_summary
)

,period_type,rows
0,COMPARATIVE,89731
1,CURRENT,89709
2,VALUE_MISSING,25593


In [165]:
corrected_status_summary = (
    period_mapped_corrected_df[
        "mapping_status"
    ]
    .value_counts(
        dropna=False
    )
    .reset_index()
)

corrected_status_summary.columns = [
    "mapping_status",
    "rows"
]

display(
    corrected_status_summary
)

,mapping_status,rows
0,DATE_HEADER_PAIR,124784
1,EXPLICIT_TOKEN,54563
2,LABEL_FOUND_VALUE_MISSING,25593
3,DATE_HEADER_SINGLE_COMPARATIVE,54
4,DATE_HEADER_SINGLE_CURRENT,39


In [166]:
corrected_unknown_df = (
    period_mapped_corrected_df[
        period_mapped_corrected_df[
            "period_type"
        ].eq(
            "UNKNOWN"
        )
    ]
    .copy()
)

print(
    "UNKNOWN after correction:",
    len(corrected_unknown_df)
)

display(
    corrected_unknown_df[
        [
            "ticker",
            "year",
            "quarter",
            "metric",
            "period_header",
            "parsed_period_date",
            "expected_period_date",
            "mapping_status",
            "source_file",
            "source_sheet"
        ]
    ].head(100)
)

UNKNOWN after correction: 0


,ticker,year,quarter,metric,period_header,parsed_period_date,expected_period_date,mapping_status,source_file,source_sheet


In [167]:
from pathlib import Path

FINAL_PERIOD_FILE = Path(
    "data/idx_financial/period_mapping/idx_financial_period_mapped_corrected.csv"
)

period_mapped_corrected_df.to_csv(
    FINAL_PERIOD_FILE,
    index=False
)

print(
    "Saved:",
    FINAL_PERIOD_FILE
)

print(
    "Rows:",
    len(period_mapped_corrected_df)
)

Saved: data\idx_financial_period_mapped_corrected.csv
Rows: 205033
